## Setup

In [3]:
import os

import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

# L?dt Variablen aus einer lokalen .env Datei (wenn vorhanden)
load_dotenv(override=False)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = (
    os.getenv('SUPABASE_ANON_KEY')
    or os.getenv('SUPABASE_KEY')
    or os.getenv('SUPABASE_SERVICE_ROLE_KEY')
)

if not SUPABASE_URL or not SUPABASE_KEY:
    raise RuntimeError(
        'Supabase-Credentials fehlen. Lege eine .env Datei im Repo an mit:\n'
        'SUPABASE_URL=...\n'
        'SUPABASE_ANON_KEY=... (oder SUPABASE_SERVICE_ROLE_KEY=...)'
    )

sb = create_client(SUPABASE_URL, SUPABASE_KEY)
print('? Supabase Client initialisiert')


? Supabase Client initialisiert


In [4]:
# Test: Zugriff auf Tabelle 'archive' (erste 5 Zeilen)
resp = sb.table('archive').select('*').limit(5).execute()

# supabase-py wirft teils Exceptions; falls ein error-Objekt vorhanden ist, zeigen wir es explizit
err = getattr(resp, 'error', None)
if err:
    raise RuntimeError(str(err))

rows = resp.data or []
df = pd.DataFrame(rows)
df


,id,simap_project_id,simap_publication_id,project_number,publication_number,title_de,title_fr,description_de,description_fr,publication_date,...,total_price_selection,has_project_documents,raw_json_search,raw_json_detail,created_at,updated_at,detail_fetched_at,detail_fetch_error,content_hash,last_checked_at
0,29131d52-9bca-4b9c-9cd8-7baf74fb7448,565,315531,None,None,2'000'000 Portionen Militärschokolade,None,None,None,2008-07-31,...,None,False,"{'id': 315531, 'bkp': None, 'cpv': '15842100, ...","{'ID': '315531', 'LANG': 'DE', 'OB01': {'OB.WT...",2026-04-14T15:09:29.935507+00:00,2026-04-15T10:15:35+00:00,2026-04-15T10:15:35+00:00,None,85f9046239fbcda2ce071c6ca1901332,None
1,ab38ccf0-81d5-4831-ace9-77df06fe9ae0,35686,497169,None,None,"N02, EP Schänzli, Massnahmenkonzept (MK)/ Ausf...",None,None,None,2010-06-10,...,None,False,"{'id': 497169, 'bkp': None, 'cpv': '71300000',...","{'ID': '497169', 'LANG': 'DE', 'OB02': {'OB.WT...",2026-04-14T15:09:49.615508+00:00,2026-04-15T08:59:21+00:00,2026-04-15T08:59:21+00:00,None,dfc954cdf0efd0e416a7bb5c5051be8f,None
2,0dcb6817-9711-4219-a7d4-1b00fdb6c0ab,95171,792567,None,None,"UW Oerlikon Neu, Knicktore Netzstützpunkt",None,None,None,2013-10-04,...,None,False,"{'id': 792567, 'bkp': None, 'cpv': '44221300',...","{'ID': '792567', 'LANG': 'DE', 'OB02': {'OB.WT...",2026-04-14T15:14:10.69272+00:00,2026-04-16T11:05:05+00:00,2026-04-16T11:05:05+00:00,None,4e9bd575907adc16f807240b45d07659,None
3,220ee6e1-efad-4cdd-b738-5658f565354a,102249,793023,None,None,"IWB Unterwerk Jakobsberg, BKP 244, BKP 246 Lüf...",None,None,None,2013-10-05,...,None,False,"{'id': 793023, 'bkp': '244, 246', 'cpv': '4251...","{'ID': '793023', 'LANG': 'DE', 'OB02': {'OB.WT...",2026-04-14T15:14:10.69272+00:00,2026-04-16T11:05:06+00:00,2026-04-16T11:05:06+00:00,None,062e76fd209ef35bf8d934fb366c3a39,None
4,45e114f3-3b53-4d96-a120-ecf039adae42,84458,792663,None,None,None,"IHEID, Maison de la Paix. Dossier d'appel d'of...",None,None,2013-10-08,...,None,False,"{'id': 792663, 'bkp': None, 'cpv': '30232110',...","{'ID': '792663', 'LANG': 'FR', 'OB02': {'OB.WT...",2026-04-14T15:14:10.69272+00:00,2026-04-16T11:05:05+00:00,2026-04-16T11:05:05+00:00,None,7bc0eb2218f37ef9991f2651732a2ff5,None


---
## Explorative Datenanalyse (EDA)

Nach der Bereinigung in Supabase (siehe `data_transformation_log.txt`):
- Duplikate pro `(simap_project_id, pub_type, creation_language)` entfernt
- Nicht-enrichte Einträge gelöscht

Die folgenden Zellen laden die bereinigte `archive`-Tabelle und zeigen
Data-Science-typische Diagnosen, Verteilungen und Zeitreihen.

### 1) Daten laden
Wir laden die bereinigte `archive`-Tabelle paginiert ueber `supabase-py`.
Grosse JSON-Spalten (`raw_json_search`, `raw_json_detail`) werden bewusst
ausgelassen, um Speicher zu sparen.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 12

ANALYSE_SPALTEN = [
    'id', 'simap_project_id', 'simap_publication_id',
    'publication_date', 'award_decision_date',
    'pub_type', 'project_type', 'process_type', 'order_type',
    'creation_language',
    'canton', 'proc_office_canton', 'winner_canton',
    'cpv_code_main',
    'award_amount', 'award_currency',
    'number_of_submissions', 'lots_count',
    'winner_name', 'proc_office_name_de',
    'has_project_documents',
    'detail_fetched_at',
]

def lade_archiv(spalten=ANALYSE_SPALTEN, batch=5000):
    alle, offset = [], 0
    while True:
        resp = (
            sb.table('archive')
              .select(','.join(spalten))
              .order('publication_date')
              .range(offset, offset + batch - 1)
              .execute()
        )
        rows = resp.data or []
        if not rows:
            break
        alle.extend(rows)
        if len(rows) < batch:
            break
        offset += batch
    df = pd.DataFrame(alle)
    for col in ['publication_date', 'award_decision_date', 'detail_fetched_at']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_localize(None)
    for col in ['award_amount']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

df = lade_archiv()
print(f'Geladen: {len(df):,} Zeilen / {df.shape[1]} Spalten')
df.head(3)


ModuleNotFoundError: No module named 'matplotlib'

### 2) Uebersicht & Datenqualitaet
Shape, Datentypen, Speicherbedarf, Anteil fehlender Werte je Spalte.
Das ist der klassische *Data-Science Sanity-Check* nach dem Laden.

In [ ]:
print('Shape:', df.shape)
print(f'Speicher: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Zeitraum publication_date:',
      df['publication_date'].min(), '->', df['publication_date'].max())
print('Eindeutige Auftraege (simap_project_id):',
      df['simap_project_id'].nunique())
print('Eindeutige Publikationen:',
      df['simap_publication_id'].nunique())
df.dtypes.to_frame('dtype').assign(
    non_null=df.notna().sum().values,
    missing_pct=(df.isna().mean().values * 100).round(2),
)


In [ ]:
missing = df.isna().mean().sort_values(ascending=True) * 100
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(9, max(4, 0.3 * len(missing))))
ax.barh(missing.index, missing.values, color=sns.color_palette('rocket_r', len(missing)))
ax.set_xlabel('Fehlende Werte (%)')
ax.set_title('Fehlwert-Anteil je Spalte (Analyse-Subset)')
for i, v in enumerate(missing.values):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=8)
plt.tight_layout(); plt.show()


### 3) Zeitliche Verteilung
Wie viele Publikationen pro Jahr? Wie verteilen sich Ausschreibungen
(`OB01`) vs. Zuschlaege (`OB02`)?

In [ ]:
df['year'] = df['publication_date'].dt.year
jahresstat = df.groupby(['year', 'pub_type']).size().unstack(fill_value=0)
topt = df['pub_type'].value_counts().head(4).index.tolist()
jahresstat = jahresstat[topt]

fig, ax = plt.subplots(figsize=(12, 5))
jahresstat.plot.area(ax=ax, alpha=0.85, colormap='viridis')
ax.set_title('Publikationen pro Jahr, aufgeteilt nach pub_type')
ax.set_ylabel('Anzahl Publikationen'); ax.set_xlabel('Jahr')
ax.legend(title='pub_type', loc='upper left')
plt.tight_layout(); plt.show()

jahresstat.tail(10)


In [ ]:
df['month'] = df['publication_date'].dt.month
pivot = (df.dropna(subset=['year', 'month'])
           .groupby(['year', 'month']).size().unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(12, max(4, 0.22 * len(pivot))))
sns.heatmap(pivot, cmap='YlOrRd', cbar_kws={'label': 'Publikationen'}, ax=ax)
ax.set_title('Heatmap: Publikationen pro Jahr und Monat')
ax.set_xlabel('Monat'); ax.set_ylabel('Jahr')
plt.tight_layout(); plt.show()


### 4) Kategorische Verteilungen
`pub_type`, `process_type`, `order_type`, `project_type` geben den
fachlichen Kontext der Ausschreibungen.

In [ ]:
cat_cols = ['pub_type', 'process_type', 'order_type', 'project_type']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_cols):
    vc = df[col].fillna('(NULL)').value_counts().head(10)
    sns.barplot(x=vc.values, y=vc.index, ax=ax,
                palette='mako', hue=vc.index, legend=False)
    ax.set_title(f'Top 10 {col}')
    ax.set_xlabel('Anzahl')
plt.tight_layout(); plt.show()


### 5) Geografische Verteilung
Publikationen je Kanton (Beschaffungsstelle) und Gewinner-Kanton.
Beachte: `canton` ist haeufig NULL, wir nutzen daher
`proc_office_canton`.

In [ ]:
kanton = df['proc_office_canton'].fillna('??').value_counts()
winner = df['winner_canton'].fillna('??').value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.barplot(x=kanton.head(15).values, y=kanton.head(15).index, ax=axes[0],
            palette='crest', hue=kanton.head(15).index, legend=False)
axes[0].set_title('Top 15 Beschaffungsstellen-Kantone'); axes[0].set_xlabel('Publikationen')

sns.barplot(x=winner.head(15).values, y=winner.head(15).index, ax=axes[1],
            palette='flare', hue=winner.head(15).index, legend=False)
axes[1].set_title('Top 15 Gewinner-Kantone'); axes[1].set_xlabel('Zuschlaege')
plt.tight_layout(); plt.show()


### 6) Zuschlagssummen (`award_amount`)
Preisverteilung klassisch in *Log-Skala*, da sehr schief.
Zusaetzlich: Perzentil-Tabelle und Boxplot pro Jahr (Median).

In [ ]:
awd = df.loc[df['award_amount'].notna() & (df['award_amount'] > 0), 'award_amount']
print(f'Zuschlaege mit Betrag: {len(awd):,}')
perc = awd.describe(percentiles=[.01,.1,.25,.5,.75,.9,.99])
print(perc.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(np.log10(awd), bins=60, color='#3a7ca5', edgecolor='white')
axes[0].set_title('Verteilung log10(award_amount)')
axes[0].set_xlabel('log10 CHF'); axes[0].set_ylabel('Haeufigkeit')

sns.boxplot(data=df.assign(log_amount=np.log10(df['award_amount'].where(df['award_amount']>0))),
            x='year', y='log_amount', ax=axes[1],
            color='#e07a5f', fliersize=1)
axes[1].set_title('log10(award_amount) pro Jahr')
axes[1].set_xlabel('Jahr'); axes[1].set_ylabel('log10 CHF')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()


### 7) Top Gewinner & Beschaffungsstellen
Wer erhaelt am meisten Zuschlaege nach Anzahl und nach Summe?

In [ ]:
zuschlaege = df[df['pub_type'] == 'OB02']

anz_winner = (zuschlaege['winner_name'].dropna().value_counts().head(15))
summe_winner = (zuschlaege.dropna(subset=['winner_name','award_amount'])
                         .groupby('winner_name')['award_amount']
                         .sum().sort_values(ascending=False).head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(x=anz_winner.values, y=anz_winner.index, ax=axes[0],
            palette='Blues_r', hue=anz_winner.index, legend=False)
axes[0].set_title('Top 15 Gewinner nach Anzahl Zuschlaege'); axes[0].set_xlabel('Anzahl')

sns.barplot(x=summe_winner.values / 1e6, y=summe_winner.index, ax=axes[1],
            palette='Greens_r', hue=summe_winner.index, legend=False)
axes[1].set_title('Top 15 Gewinner nach Summe (Mio. CHF)'); axes[1].set_xlabel('Mio. CHF')
plt.tight_layout(); plt.show()


In [ ]:
proc = (df['proc_office_name_de'].dropna().value_counts().head(15))
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(x=proc.values, y=proc.index, ax=ax,
            palette='Purples_r', hue=proc.index, legend=False)
ax.set_title('Top 15 Beschaffungsstellen nach Anzahl Publikationen')
ax.set_xlabel('Publikationen')
plt.tight_layout(); plt.show()


### 8) Korrelation numerischer Features
Heatmap ueber `award_amount` (log), `number_of_submissions`, `lots_count`
und aus `publication_date` abgeleitete Features.

In [ ]:
num = df[['award_amount', 'number_of_submissions', 'lots_count']].copy()
num['log_award_amount'] = np.log10(num['award_amount'].where(num['award_amount'] > 0))
num['year'] = df['publication_date'].dt.year
num['month'] = df['publication_date'].dt.month
num['has_project_documents_i'] = df['has_project_documents'].astype('float')

corr = num.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='vlag', center=0, fmt='.2f', ax=ax,
            linewidths=.5, cbar_kws={'shrink': .8})
ax.set_title('Korrelationsmatrix numerischer Features')
plt.tight_layout(); plt.show()


### 9) CPV-Top-Kategorien
Die fuehrende CPV-Hauptklassifikation (erste 2 Stellen)
gibt einen Beschaffungs-Branchen-Ueberblick.

In [ ]:
cpv = df['cpv_code_main'].dropna().astype(str).str[:2]
cpv_vc = cpv.value_counts().head(15)
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=cpv_vc.index, y=cpv_vc.values, ax=ax,
            palette='cubehelix', hue=cpv_vc.index, legend=False)
ax.set_title('Top 15 CPV-Hauptkategorien (2-stellig)')
ax.set_xlabel('CPV-Prefix'); ax.set_ylabel('Publikationen')
plt.tight_layout(); plt.show()


### 10) Backup-Tabelle als CSV exportieren
Die in Supabase angelegte Backup-Tabelle `archive_deleted_backup_20260416`
enthaelt alle geloeschten Zeilen (inkl. `deletion_reason`, `deleted_at`).

Der folgende Block laedt sie paginiert und schreibt `archive_deleted_backup_20260416.csv`
ins Repo-Verzeichnis. Bei 20'694 Zeilen dauert das einige Sekunden.

In [ ]:
BACKUP_TABELLE = 'archive_deleted_backup_20260416'
CSV_PFAD = f'{BACKUP_TABELLE}.csv'

cols_backup = [c for c in ANALYSE_SPALTEN if c != 'id'] + [
    'id', 'title_de', 'title_fr', 'deletion_reason', 'deleted_at'
]

def lade_backup(batch=2000):
    alle, offset = [], 0
    while True:
        resp = (sb.table(BACKUP_TABELLE)
                  .select(','.join(cols_backup))
                  .range(offset, offset + batch - 1)
                  .execute())
        rows = resp.data or []
        if not rows:
            break
        alle.extend(rows)
        if len(rows) < batch:
            break
        offset += batch
    return pd.DataFrame(alle)

backup_df = lade_backup()
backup_df.to_csv(CSV_PFAD, index=False, encoding='utf-8')
print(f'{len(backup_df):,} Zeilen gesichert -> {CSV_PFAD}')
backup_df['deletion_reason'].value_counts()
